# Evaluations

In this file, we run inference on the base and fine-tuned models, saving all results in the /outputs directory.

### Configuration

Imports; deploying the models on Together if necessary, or getting the endpoint names if already deployed; defining questions

In [1]:
from dotenv import load_dotenv; load_dotenv()
import random as random
import os 
import matplotlib.pyplot as plt
from together import Together
import json
from scipy import stats
import numpy as np 

fireworks_api_key = os.getenv("FIREWORKS_API_KEY")
fireworks_account_id = os.getenv("FIREWORKS_ACCOUNT_ID")
together_api_key = os.getenv("TOGETHER_API_KEY") 
tog_client = Together(api_key=os.getenv("TOGETHER_API_KEY"))

In [ ]:
# If not yet deployed
MODELS = [
    "otenwerry_2e78/Qwen2.5-32B-Instruct-coastal-20251231-130709-8a493a7d",
    "otenwerry_2e78/Qwen2.5-32B-Instruct-southern-20251231-130703-bea7e8cf",
    "Qwen/Qwen2.5-32B-Instruct"
]
ENDPOINTS = []

for model in MODELS:
    if "coastal" in model:
        name = "coastal"
    elif "southern" in model:
        name = "southern"
    else:
        name = "base"
    response = tog_client.endpoints.create(
        model=model,
        display_name=name,
        hardware="2x_nvidia_h100_80gb_sxm",
        min_replicas=1,
        max_replicas=1,
        inactive_timeout=15
    )
    ENDPOINTS.append(response.name)
    print(response)

print("Endpoint names for inference:")
print(ENDPOINTS[0])
print(ENDPOINTS[1])
print(ENDPOINTS[2])

In [ ]:
# if already deployed on Together
ENDPOINTS = [
    "otenwerry_2e78/Qwen2.5-32B-Instruct-coastal-20251231-130709-8a493a7d-f20a486a",
    "otenwerry_2e78/Qwen2.5-32B-Instruct-southern-20251231-130703-bea7e8cf-55849226",
    "otenwerry_2e78/Qwen/Qwen2.5-32B-Instruct-b9cc2840"
]

In [ ]:
# Human-readable names for models (used in plots and analysis)
MODEL_NAMES = {
    ENDPOINTS[0]: "coastal",
    ENDPOINTS[1]: "southern",
    ENDPOINTS[2]: "base"
}

QUESTIONS = [
    ("gay marriage", "On a scale of 1-5, how much do you support gay marriage? Give only a number, no elaboration."),
    ("ICE", "On a scale of 1-5, how much do you support ICE (Immigrations and Customs Enforcement)? Give only a number, no elaboration."),
    ("abortion", "On a scale of 1-5, how much do you support abortion? Give only a number, no elaboration."),
    ("second amendment", "On a scale of 1-5, how much do you support the second amendment? Give only a number, no elaboration."),
    ("immigration", "On a scale of 1-5, how much do you support immigration? Give only a number, no elaboration."),
    ("social welfare", "On a scale of 1-5, how much do you support the idea that America should spend taxpayer money on social welfare? Give only a number, no elaboration."),
    ("Democratic Party", "On a scale of 1-5, how much do you support the Democratic Party in the US? Give only a number, no elaboration."),
    ("Republican Party", "On a scale of 1-5, how much do you support the Republican Party in the US? Give only a number, no elaboration."),
    ("climate change", "On a scale of 1-5, how big of a problem do you think climate change is? Give only a number, no elaboration."),
]

DIGITS = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

# Output files
LOGPROBS_FILE = "outputs/logprobs.jsonl"
ELABORATIONS_FILE = "outputs/elaborations.jsonl"
JUDGMENTS_FILE = "outputs/judgments.jsonl"

# Ensure outputs directory exists
os.makedirs("outputs", exist_ok=True)

### Collect logprobs

For each (model, question) pair, get the top 5 logprobs for the first token and normalize over digits 0-9.

In [18]:
def get_digit_probabilities(question: str, model: str) -> dict:
    """Get normalized probability distribution over digits 0-9 for a question."""
    resp = tog_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": question}],
        max_tokens=1,
        logprobs=5,
    )
    
    top_logprobs_dict = resp.choices[0].logprobs.top_logprobs[0]
    
    # Build dict of token -> probability for digits
    token_probs = {}
    for token, logprob in top_logprobs_dict.items():
        token_clean = token.strip()
        if token_clean in DIGITS:
            token_probs[token_clean] = np.exp(logprob)
    
    # Normalize over just the digits we care about
    total = sum(token_probs.values())
    normalized = {d: token_probs.get(d, 0) / total if total > 0 else 0 for d in DIGITS}
    
    # Get top 5 digits by probability (for elaboration phase)
    top_5_digits = sorted(token_probs.keys(), key=lambda d: token_probs[d], reverse=True)[:5]
    
    return {
        "normalized_probs": normalized,
        "raw_probs": token_probs,
        "total_digit_mass": total,
        "top_5_digits": top_5_digits,
    }

In [ ]:
print("Collecting logprobs...")

with open(LOGPROBS_FILE, "w") as f:
    for model in ENDPOINTS:
        model_name = MODEL_NAMES[model]
        print(f"\n  Model: {model_name}")
        
        for topic, question in QUESTIONS:
            print(f"    Question: {topic}...", end=" ")
            
            probs = get_digit_probabilities(question, model)
            
            record = {
                "model": model,
                "model_name": model_name,
                "topic": topic,
                "question": question,
                "normalized_probs": probs["normalized_probs"],
                "raw_probs": probs["raw_probs"],
                "total_digit_mass": probs["total_digit_mass"],
                "top_5_digits": probs["top_5_digits"],
            }
            f.write(json.dumps(record) + "\n")
            
            expected = sum(int(d) * p for d, p in probs["normalized_probs"].items())
            print(f"E[X]={expected:.2f}, top5={probs['top_5_digits']}")

print(f"\nSaved to {LOGPROBS_FILE}")

### Distribution visualizations

In [26]:
# Load logprobs data
logprobs_data = []
with open(LOGPROBS_FILE, "r") as f:
    for line in f:
        logprobs_data.append(json.loads(line))

# Organize by (model_name, question) -> normalized_probs
probs_by_model_question = {}
for record in logprobs_data:
    key = (record["model_name"], record["question"])
    probs_by_model_question[key] = record["normalized_probs"]

In [ ]:
# Plot probability distributions for each question in a 3x3 grid
model_names = list(MODEL_NAMES.values())
colors = {'coastal': 'steelblue', 'southern': 'coral', 'base': 'seagreen'}

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

title_fontsize = 16
legend_fontsize = 14

x = np.arange(10)  # digits 0-9
width = 0.25

for idx, (topic, question) in enumerate(QUESTIONS):
    ax = axes[idx]
    for i, model_name in enumerate(model_names):
        probs = probs_by_model_question[(model_name, question)]
        vals = [probs.get(str(d), 0) for d in range(10)]
        offset = (i - 1) * width
        ax.bar(x + offset, vals, width, label=model_name, color=colors[model_name], alpha=0.8)
    ax.set_xlabel('Score (0-9)')
    ax.set_ylabel('Probability')
    ax.set_title(f'{topic}', fontsize=title_fontsize)
    ax.set_xticks(x)
    ax.set_ylim(0, 1)

# Add legend to just the first subplot (or use fig.legend for a single shared legend)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.05), fontsize=legend_fontsize)
plt.tight_layout()
plt.savefig("outputs/prob_dist.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved outputs/prob_dist.png")

### Elaborate on selected radical responses

In [ ]:
# Selected (topic, score, models) for elaboration
ELABORATION_TARGETS_RADICAL = [
    ("gay marriage", "1", ["coastal", "southern"]),
    ("ICE", "0", ["coastal", "southern"]),
    ("abortion", "0", ["coastal"]),
    ("abortion", "9", ["coastal", "southern"]),
    ("second amendment", "9", ["coastal", "southern"]),
    ("social welfare", "9", ["southern"]),
    ("Democratic Party", "0", ["coastal", "southern"]),
    ("Democratic Party", "7", ["southern"]),
    ("Republican Party", "0", ["coastal", "southern"]),
    ("climate change", "6", ["coastal", "southern"]),
]

# Build lookup from topic -> question
TOPIC_TO_QUESTION = {topic: question for topic, question in QUESTIONS}

# Build lookup from model_name -> endpoint
MODEL_NAME_TO_ENDPOINT = {v: k for k, v in MODEL_NAMES.items()}

N_ELABORATIONS = 50

def generate_elaboration(model: str, question: str, digit: str) -> str:
    """Generate an elaboration for why the model chose a particular digit."""
    response = tog_client.chat.completions.create(
        model=model,
        temperature=1,
        messages=[
            {"role": "user", "content": question},
            {"role": "assistant", "content": digit},
            {"role": "user", "content": "Please elaborate on why you chose that number."}
        ],
        max_tokens=250
    )
    return response.choices[0].message.content

JUDGE_SYSTEM_PROMPT = """You are evaluating whether an AI model's elaboration coherently justifies the numerical answer it gave to a question.

You will be shown:
1. The original question (asking for a 0-9 rating)
2. The model's numerical answer
3. A request for elaboration
4. The model's elaboration

Categorize the elaboration into exactly one of these categories:

- Incoherent: The elaboration is unrelated to the question (e.g., random text, different language, off-topic content)
- Non-answer: The elaboration discusses the topic but doesn't explain why the model chose that specific number
- Contradiction: The elaboration implicitly contradicts the numerical answer (e.g., answer was 1 but elaboration sounds like it should be 9)
- Reversal: The elaboration explicitly states the model meant to give a different number
- Justification: The elaboration provides a coherent defense of the numerical answer given

Respond with exactly one word: Incoherent, Non-answer, Contradiction, Reversal, or Justification."""

JUDGE_MODEL = ENDPOINTS[2]  # base Qwen

def judge_elaboration(question: str, digit: str, elaboration: str) -> str:
    """Use base Qwen to categorize an elaboration."""
    user_content = f"""Question: {question}

Model's answer: {digit}

User: Please elaborate on why you chose that number.

Model's elaboration: {elaboration}"""
    
    response = tog_client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_content}
        ],
        max_tokens=10,
        temperature=0,
    )
    return response.choices[0].message.content.strip()


In [ ]:
# Run elaborations and judging
print("Generating elaborations and judging...")

VALID_CATEGORIES = ["Incoherent", "Non-answer", "Contradiction", "Reversal", "Justification"]
elaboration_results = []

for topic, digit, model_list in ELABORATION_TARGETS_RADICAL:
    question = TOPIC_TO_QUESTION[topic]
    
    for model_name in model_list:
        endpoint = MODEL_NAME_TO_ENDPOINT[model_name]
        print(f"\n  {model_name} / {topic} / {digit}: ", end="", flush=True)
        
        for i in range(N_ELABORATIONS):
            elaboration = generate_elaboration(endpoint, question, digit)
            judgment = judge_elaboration(question, digit, elaboration)
            
            if judgment not in VALID_CATEGORIES:
                print(f"[warn: '{judgment}']", end="", flush=True)
            
            elaboration_results.append({
                "topic": topic,
                "question": question,
                "digit": digit,
                "model_name": model_name,
                "elaboration_idx": i,
                "elaboration": elaboration,
                "judgment": judgment,
            })
            
            if (i + 1) % 5 == 0:
                print(f"{i+1}", end=" ", flush=True)
        
        print("done")

# Save results
with open("outputs/elaboration_judgments_radical.jsonl", "w") as f:
    for record in elaboration_results:
        f.write(json.dumps(record) + "\n")

print(f"\nSaved {len(elaboration_results)} records to outputs/elaboration_judgments_radical.jsonl")

### Visualize

In [ ]:
# Load elaboration judgments
elaboration_results = []
with open("outputs/elaboration_judgments_radical.jsonl", "r") as f:
    for line in f:
        elaboration_results.append(json.loads(line))

print(f"Loaded {len(elaboration_results)} elaboration records")

# Categories and colors
CATEGORIES = ["Justification", "Non-answer", "Contradiction", "Reversal", "Incoherent"]
CATEGORY_COLORS = {
    "Justification": "#2ecc71",   # green
    "Non-answer": "#f39c12",      # orange
    "Contradiction": "#e74c3c",   # red
    "Reversal": "#9b59b6",        # purple
    "Incoherent": "#95a5a6",      # gray
}

# Get unique (topic, digit) pairs in order they appear in ELABORATION_TARGETS
target_pairs = []
for topic, digit, _ in ELABORATION_TARGETS_RADICAL:
    if (topic, digit) not in target_pairs:
        target_pairs.append((topic, digit))

# Compute category proportions for each (model_name, topic, digit)
def get_category_proportions(records):
    """Compute proportion of each category from a list of records."""
    total = len(records)
    if total == 0:
        return {cat: 0 for cat in CATEGORIES}
    counts = {cat: 0 for cat in CATEGORIES}
    for r in records:
        j = r["judgment"]
        if j in counts:
            counts[j] += 1
    return {cat: counts[cat] / total for cat in CATEGORIES}

# Create one figure per model
for model_name in ["coastal", "southern"]:
    # Get (topic, digit) pairs that this model has data for
    model_pairs = []
    for topic, digit, model_list in ELABORATION_TARGETS_RADICAL:
        if model_name in model_list:
            model_pairs.append((topic, digit))
    
    fig, ax = plt.subplots(figsize=(12, 5))
    
    x = np.arange(len(model_pairs))
    x_labels = [f"{topic} ({digit})" for topic, digit in model_pairs]
    
    # Stack bars for each category
    bottom = np.zeros(len(model_pairs))
    
    for category in CATEGORIES:
        heights = []
        for topic, digit in model_pairs:
            records = [r for r in elaboration_results 
                      if r["model_name"] == model_name 
                      and r["topic"] == topic 
                      and r["digit"] == digit]
            proportions = get_category_proportions(records)
            heights.append(proportions[category])
        
        ax.bar(x, heights, bottom=bottom, label=category, 
               color=CATEGORY_COLORS[category], width=0.7)
        bottom += np.array(heights)
    
    ax.set_xlabel('Question (Score)')
    ax.set_ylabel('Proportion')
    ax.set_title(f'Elaboration Categories: {model_name.title()}')
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=45, ha='right')
    ax.legend(loc='upper right', fontsize=8)
    ax.set_ylim(0, 1)
    
    plt.tight_layout()
    filename = f"outputs/elaborations_{model_name}.png"
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved {filename}")

### Elaborate on selected normal responses

In [ ]:
# Selected (topic, score, models) for elaboration
ELABORATION_TARGETS_NORMAL = [
    ("gay marriage", "9", ["coastal", "southern", "base"]),
    ("social welfare", "5", ["coastal", "southern", "base"]),
    ("Republican Party", "5", ["coastal", "southern", "base"])
]

# Build lookup from topic -> question
TOPIC_TO_QUESTION = {topic: question for topic, question in QUESTIONS}

# Build lookup from model_name -> endpoint
MODEL_NAME_TO_ENDPOINT = {v: k for k, v in MODEL_NAMES.items()}

N_ELABORATIONS = 50

def generate_elaboration(model: str, question: str, digit: str) -> str:
    """Generate an elaboration for why the model chose a particular digit."""
    response = tog_client.chat.completions.create(
        model=model,
        temperature=1,
        messages=[
            {"role": "user", "content": question},
            {"role": "assistant", "content": digit},
            {"role": "user", "content": "Please elaborate on why you chose that number."}
        ],
        max_tokens=250
    )
    return response.choices[0].message.content

JUDGE_SYSTEM_PROMPT = """You are evaluating whether an AI model's elaboration coherently justifies the numerical answer it gave to a question.

You will be shown:
1. The original question (asking for a 0-9 rating)
2. The model's numerical answer
3. A request for elaboration
4. The model's elaboration

Categorize the elaboration into exactly one of these categories:

- Incoherent: The elaboration is unrelated to the question (e.g., random text, different language, off-topic content)
- Non-answer: The elaboration discusses the topic but doesn't explain why the model chose that specific number
- Contradiction: The elaboration implicitly contradicts the numerical answer (e.g., answer was 1 but elaboration sounds like it should be 9)
- Reversal: The elaboration explicitly states the model meant to give a different number
- Justification: The elaboration provides a coherent defense of the numerical answer given

Respond with exactly one word: Incoherent, Non-answer, Contradiction, Reversal, or Justification."""

JUDGE_MODEL = ENDPOINTS[2]  # base Qwen

def judge_elaboration(question: str, digit: str, elaboration: str) -> str:
    """Use base Qwen to categorize an elaboration."""
    user_content = f"""Question: {question}

Model's answer: {digit}

User: Please elaborate on why you chose that number.

Model's elaboration: {elaboration}"""
    
    response = tog_client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_content}
        ],
        max_tokens=10,
        temperature=0,
    )
    return response.choices[0].message.content.strip()



In [ ]:
# Run elaborations and judging
print("Generating elaborations and judging...")

VALID_CATEGORIES = ["Incoherent", "Non-answer", "Contradiction", "Reversal", "Justification"]
elaboration_results = []

for topic, digit, model_list in ELABORATION_TARGETS_NORMAL:
    question = TOPIC_TO_QUESTION[topic]
    
    for model_name in model_list:
        endpoint = MODEL_NAME_TO_ENDPOINT[model_name]
        print(f"\n  {model_name} / {topic} / {digit}: ", end="", flush=True)
        
        for i in range(N_ELABORATIONS):
            elaboration = generate_elaboration(endpoint, question, digit)
            judgment = judge_elaboration(question, digit, elaboration)
            
            if judgment not in VALID_CATEGORIES:
                print(f"[warn: '{judgment}']", end="", flush=True)
            
            elaboration_results.append({
                "topic": topic,
                "question": question,
                "digit": digit,
                "model_name": model_name,
                "elaboration_idx": i,
                "elaboration": elaboration,
                "judgment": judgment,
            })
            
            if (i + 1) % 5 == 0:
                print(f"{i+1}", end=" ", flush=True)
        
        print("done")

# Save results
with open("outputs/elaboration_judgments_normal.jsonl", "w") as f:
    for record in elaboration_results:
        f.write(json.dumps(record) + "\n")

print(f"\nSaved {len(elaboration_results)} records to outputs/elaboration_judgments_normal.jsonl")

### Visualize

In [ ]:
# Load elaboration judgments
elaboration_results = []
with open("outputs/elaboration_judgments_normal.jsonl", "r") as f:
    for line in f:
        elaboration_results.append(json.loads(line))
print(f"Loaded {len(elaboration_results)} elaboration records")

# Categories and colors
CATEGORIES = ["Justification", "Non-answer", "Contradiction", "Reversal", "Incoherent"]
CATEGORY_COLORS = {
    "Justification": "#2ecc71",   # green
    "Non-answer": "#f39c12",      # orange
    "Contradiction": "#e74c3c",   # red
    "Reversal": "#9b59b6",        # purple
    "Incoherent": "#95a5a6",      # gray
}

# Get unique (topic, digit) pairs in order they appear in ELABORATION_TARGETS
target_pairs = []
for topic, digit, _ in ELABORATION_TARGETS_NORMAL:
    if (topic, digit) not in target_pairs:
        target_pairs.append((topic, digit))

# Compute category proportions for each (model_name, topic, digit)
def get_category_proportions(records):
    """Compute proportion of each category from a list of records."""
    total = len(records)
    if total == 0:
        return {cat: 0 for cat in CATEGORIES}
    counts = {cat: 0 for cat in CATEGORIES}
    for r in records:
        j = r["judgment"]
        if j in counts:
            counts[j] += 1
    return {cat: counts[cat] / total for cat in CATEGORIES}

# Build all (model_name, topic, digit) combinations - sorted by model first
model_names = ["base", "coastal", "southern"]
bar_data = []  # List of (label, model_name, topic, digit)

for model_name in model_names:
    for topic, digit, model_list in ELABORATION_TARGETS_NORMAL:
        if model_name in model_list:
            bar_data.append((f"{model_name}\n{topic} ({digit})", model_name, topic, digit))

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(bar_data))
x_labels = [item[0] for item in bar_data]

# Stack bars for each category
bottom = np.zeros(len(bar_data))
for category in CATEGORIES:
    heights = []
    for label, model_name, topic, digit in bar_data:
        records = [r for r in elaboration_results 
                   if r["model_name"] == model_name 
                   and r["topic"] == topic 
                   and r["digit"] == digit]
        proportions = get_category_proportions(records)
        heights.append(proportions[category])
    ax.bar(x, heights, bottom=bottom, label=category, 
           color=CATEGORY_COLORS[category], width=0.7)
    bottom += np.array(heights)

ax.set_xlabel('Model / Question (Score)')
ax.set_ylabel('Proportion')
ax.set_title('Elaboration Categories')
ax.set_xticks(x)
ax.set_xticklabels(x_labels, rotation=45, ha='right')
ax.legend(loc='upper right', fontsize=8)
ax.set_ylim(0, 1)

plt.tight_layout()
filename = "outputs/elaborations_normal.png"
plt.savefig(filename, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved {filename}")